# Music Sentiment as an Economic Signal

**Working research question:**  
Can shifts in popular music — especially sonic sentiment and genre composition — provide information that improves forecasts of future consumer sentiment?

### Current notebook status
This notebook is currently focused on:

1. Loading and auditing the Billboard Hot 100 data.
2. Loading the 550K Spotify-derived song metadata without the very large lyrics column.
3. Documenting known data-quality problems in the 550K dataset.
4. Engineering a more defensible broad genre taxonomy.
5. Preparing for Billboard-to-song matching.

### Important data decision
The 550K dataset's `year` field is **not treated as the authoritative timeline** because obvious date errors were found (for example, modern songs coded as 1900). The Billboard `Date` column will be the time/popularity backbone for the forecasting analysis.

The original notebook has been cleaned up, but exploratory code that helped reach these decisions is preserved near the bottom as commented-out "thought process" cells.

## 1. Imports and project paths

Only packages needed for the current data-preparation stage are imported below.

Packages for fuzzy matching, forecasting, diagnostics, and plotting are preserved as comments for later so the notebook does not load unnecessary dependencies yet.

In [1]:
import ast
import re
import unicodedata
from pathlib import Path

import pandas as pd

# --- Later stages: keep for reference, but not needed yet ---
# import numpy as np
# import matplotlib.pyplot as plt
#
# from rapidfuzz import fuzz, process
# from sklearn.metrics import mean_squared_error, mean_absolute_error
# from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
# from statsmodels.tsa.ar_model import AutoReg
# from statsmodels.tsa.statespace.sarimax import SARIMAX
# from statsmodels.stats.diagnostic import acorr_ljungbox

PROJECT_ROOT = Path.cwd()

# If the notebook is opened from a notebooks/ subfolder, move one level up.
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

BILLBOARD_PATH = DATA_RAW / "hot100.csv"
SONGS_PATH = DATA_RAW / "songs.csv"
SONG_METADATA_PATH = DATA_PROCESSED / "song_metadata.parquet"

print("Project root:", PROJECT_ROOT)

Project root: /Users/mercyeme/Documents/Data_Projects_2026/RecessionPop


## 2. Billboard Hot 100

The full Billboard table remains untouched as the weekly popularity/time table.

A separate `billboard_songs` table contains only unique `Song` + `Artist` pairs.  
This prevents repeatedly matching the same recording for every week it appears on the chart.

In [2]:
billboard = pd.read_csv(BILLBOARD_PATH)

print("Shape:", billboard.shape)
print("Columns:", billboard.columns.tolist())

billboard["Date"] = pd.to_datetime(billboard["Date"])

display(billboard.head())

print("First chart date:", billboard["Date"].min())
print("Last chart date:", billboard["Date"].max())

Shape: (354858, 8)
Columns: ['Date', 'Song', 'Artist', 'Rank', 'Last Week', 'Peak Position', 'Weeks in Charts', 'Image URL']


,Date,Song,Artist,Rank,Last Week,Peak Position,Weeks in Charts,Image URL
0,1958-07-30,Poor Little Fool,Ricky Nelson,1,-,1,1,NaN
1,1958-07-30,Patricia,Perez Prado And His Orchestra,2,-,2,1,NaN
2,1958-07-30,Splish Splash,Bobby Darin,3,-,3,1,NaN
3,1958-07-30,Hard Headed Woman,Elvis Presley With The Jordanaires,4,-,4,1,https://charts-static.billboard.com/img/1958/0...
4,1958-07-30,When,Kalin Twins,5,-,5,1,NaN


First chart date: 1958-07-30 00:00:00
Last chart date: 2026-07-29 00:00:00


In [3]:
billboard_songs = (
    billboard[["Song", "Artist"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(f"Weekly Billboard rows: {len(billboard):,}")
print(f"Unique song-artist pairs: {len(billboard_songs):,}")

display(billboard_songs.head())

Weekly Billboard rows: 354,858
Unique song-artist pairs: 32,592


,Song,Artist
0,Poor Little Fool,Ricky Nelson
1,Patricia,Perez Prado And His Orchestra
2,Splish Splash,Bobby Darin
3,Hard Headed Woman,Elvis Presley With The Jordanaires
4,When,Kalin Twins


## 3. Load lightweight 550K song metadata

The original `songs.csv` is very large because it contains full lyrics.

For the first pass, only metadata and audio-feature columns are loaded. If the processed Parquet file already exists, the notebook loads that instead of scanning the full CSV again.

**Lyrics stay out of memory until after Billboard matching.**

In [4]:
song_cols = [
    "id",
    "name",
    "album_name",
    "artists",
    "danceability",
    "energy",
    "key",
    "loudness",
    "mode",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo",
    "duration_ms",
    "year",
    "genre",
    "popularity",
    "total_artist_followers",
    "avg_artist_popularity",
    "artist_ids",
    "niche_genres",
]

In [5]:
if SONG_METADATA_PATH.exists():
    print("Loading processed metadata:", SONG_METADATA_PATH)
    songs = pd.read_parquet(SONG_METADATA_PATH)

else:
    print("Processed metadata not found; reading songs.csv in chunks.")

    chunks = []

    for chunk in pd.read_csv(
        SONGS_PATH,
        usecols=song_cols,
        chunksize=50_000,
    ):
        chunks.append(chunk)

    songs = pd.concat(chunks, ignore_index=True)

    songs.to_parquet(
        SONG_METADATA_PATH,
        index=False,
    )

print("Songs shape:", songs.shape)
display(songs.head(30))

Loading processed metadata: /Users/mercyeme/Documents/Data_Projects_2026/RecessionPop/data/processed/song_metadata.parquet
Songs shape: (550622, 23)


,id,name,album_name,artists,danceability,energy,key,loudness,mode,speechiness,...,valence,tempo,duration_ms,year,genre,popularity,total_artist_followers,avg_artist_popularity,artist_ids,niche_genres
0,0Prct5TDjAnEgIqbxcldY9,!,UNDEN!ABLE,"[""HELLYEAH""]",0.415,0.6050,7,-11.157,1,0.0575,...,0.193,100.059,79500,2016,Rock,0,769490,52.000000,"[""4hxDvVq5t8ebPYPdBl1F9f""]","[""groove metal"", ""metal""]"
1,2ASl4wirkeYm3OWZxXKYuq,!!,Childhood Dreams,"[""Yxngxr1""]",0.788,0.6480,7,-9.135,0,0.3150,...,0.287,79.998,114000,2019,Hip-Hop,29,143628,45.000000,"[""2jwRHcdgkRhelYEMqndDKe""]",[]
2,5tA3ImW310llKo8EMBj2Ga,!!Noble Stabbings!!,Situationist Comedy,"[""Dillinger Four""]",0.171,0.9570,2,-5.749,1,0.1490,...,0.349,175.317,197400,2002,Rock,0,36619,35.000000,"[""4YAN46l70QV0PGXlMg0iHi""]","[""melodic hardcore"", ""pop punk"", ""punk"", ""skat..."
3,0fROT4kK5oTm8xO8PX6EJF,!I'll Be Back!,!I'll Be Back!,"[""Ril\u00e8s""]",0.823,0.6120,1,-7.767,1,0.2480,...,0.688,142.959,178533,2018,Hip-Hop,43,929303,63.000000,"[""6pdcQa7by8IKuoVXvgknlI""]","[""french rap""]"
4,1xBFhv5faebv3mmwxx7DnS,!Lost!,!Lost!,"[""Ril\u00e8s""]",0.729,0.5520,7,-8.562,0,0.0650,...,0.380,86.103,186197,2018,Hip-Hop,0,929303,63.000000,"[""6pdcQa7by8IKuoVXvgknlI""]","[""french rap""]"
5,0gNNToCW3qjabgTyBSjt3H,!Que Vida! - Mono Version,Da Capo (Deluxe Edition),"[""Love""]",0.600,0.5400,9,-11.803,1,0.0328,...,0.547,125.898,220560,1966,Rock,20,273598,46.000000,"[""3Q6OOkfssqoMSTtl11J5Uk""]","[""acid rock"", ""baroque pop"", ""proto-punk"", ""ps..."
6,1SCFa6gt3r6juLsIqpdwY3,!liaF cipE,The Bree-TeenZ,"[""One Morning Left""]",0.483,0.8940,9,-4.164,1,0.0546,...,0.284,140.058,200200,2011,Rock,0,38169,34.000000,"[""4QGKWF55nqfGYToZkoZPRF""]","[""metalcore"", ""post-hardcore""]"
7,1JrpMwlzPuWgfmGiIBi9QI,""" """,Burner,"[""Breadwinner""]",0.296,0.8310,4,-8.101,1,0.0415,...,0.217,124.921,201760,1994,Rock,0,5014,16.000000,"[""3UUl93ABT7zywawX0QA7MC""]","[""math rock"", ""mathcore"", ""noise rock"", ""post-..."
8,5PIf0fmKJzgK8jgyqXaHGZ,""" R.I.P """,Escape,"[""Ram-Zet""]",0.474,0.9230,6,-4.611,1,0.1480,...,0.288,159.974,533160,2002,Rock,0,4100,7.000000,"[""41x3tK4pmcp109A09rYtxj""]","[""avant-garde"", ""gothic metal"", ""industrial me..."
9,0sZb4GBh64F0FOZzQy6QqV,"""...Because All Suffering Is Sweet to Me...""",Lucy Gray,"[""Envy On The Coast""]",0.241,0.7340,4,-6.919,0,0.0671,...,0.040,157.355,237973,2007,Rock,23,37014,31.000000,"[""6RvruFAgvsAb4B82u8wsSH""]","[""post-hardcore""]"


## 4. Data-quality notes

Two findings from exploration matter for the rest of the project:

- `year` contains obvious errors, including modern recordings assigned to 1900.
- The broad `genre` field is useful as a baseline, but some categories — especially `Folk` — contain systematic misclassifications such as reggae/dancehall.

Therefore:

- **Billboard `Date` controls time.**
- **Official broad genre is the fallback classification, not unquestioned ground truth.**
- `niche_genres` supplies evidence for targeted overrides.

In [6]:
print("550K year range:", songs["year"].min(), "to", songs["year"].max())
print("\nOfficial broad genre counts:")
display(songs["genre"].value_counts(dropna=False))

550K year range: 1900 to 2025

Official broad genre counts:


genre
Rock          197168
Pop            72539
Electronic     69755
Folk           50843
Country        46076
Hip-Hop        41745
R&B            27337
Jazz           17010
Blues          15976
Classical      12173
Name: count, dtype: int64

## 5. Parse niche genres

`niche_genres` is stored as a string representation of a Python list.  
Convert it once into an actual list so each tag can be inspected and scored safely.

In [7]:
def parse_genre_list(value):
    if isinstance(value, list):
        return value

    if pd.isna(value):
        return []

    try:
        parsed = ast.literal_eval(value)
        return parsed if isinstance(parsed, list) else []

    except (ValueError, SyntaxError, TypeError):
        return []


songs["niche_genre_list"] = (
    songs["niche_genres"]
    .apply(parse_genre_list)
)

## 6. Final broad genre taxonomy

The current working taxonomy is intentionally broad enough for monthly genre-share features:

- Pop
- Indie
- Rock
- Hip-Hop
- R&B/Soul
- Country
- Electronic/Dance
- Folk
- Reggae/Dancehall
- Latin/Tropical
- Afrobeats/African Pop
- Jazz
- Classical/Musical
- Blues
- Other

### Classification philosophy

This is **not** a manual remapping of all 754 niche tags.

Instead:

1. Start with the dataset's official broad genre.
2. Rename broad labels where needed (`R&B` → `R&B/Soul`, etc.).
3. Let high-confidence niche tags vote for a corrected category.
4. If there is one clear winning category, use it as an override.
5. If niche evidence is absent or ambiguous, keep the mapped official genre.
6. `Other` is a fallback for official labels outside the defined taxonomy — not the automatic destination for an unmapped niche tag.

This preserves the original dataset where it behaves sensibly while correcting systematic problems.

In [8]:
FINAL_GENRES = [
    "Pop",
    "Indie",
    "Rock",
    "Hip-Hop",
    "R&B/Soul",
    "Country",
    "Electronic/Dance",
    "Folk",
    "Reggae/Dancehall",
    "Latin/Tropical",
    "Afrobeats/African Pop",
    "Jazz",
    "Classical/Musical",
    "Blues",
    "Other",
]

BASE_GENRE_MAP = {
    "Pop": "Pop",
    "Rock": "Rock",
    "Hip-Hop": "Hip-Hop",
    "R&B": "R&B/Soul",
    "Country": "Country",
    "Electronic": "Electronic/Dance",
    "Folk": "Folk",
    "Jazz": "Jazz",
    "Classical": "Classical/Musical",
    "Blues": "Blues",
}

songs["genre_original"] = songs["genre"]

songs["genre_base"] = (
    songs["genre_original"]
    .map(BASE_GENRE_MAP)
    .fillna("Other")
)

## 7. High-confidence niche-tag votes

These tags provide enough musical information to vote for one of the final broad categories.

A few deliberate choices:

- generic/geographic `indie` → **Indie**
- `indie rock` → **Rock**
- `indie r&b` → **R&B/Soul**
- `indie pop` → **Pop**
- `ska` → **Reggae/Dancehall**
- `ska punk` → **Rock**
- `pop punk` → **Rock**
- `synthpop` → **Electronic/Dance**
- generic `industrial`, `darkwave`, `new wave`, and `riddim` do **not** hard-override because they are genuinely cross-genre in this dataset.
- thematic/context tags such as `christian`, `christmas`, and `worship` do not vote for a primary musical genre.

In [9]:
GENRE_VOTE_TAGS = {

    "Reggae/Dancehall": {
        "calypso",
        "christian reggae",
        "dancehall",
        "dub",
        "lovers rock",
        "nz reggae",
        "ragga",
        "reggae",
        "reggae en español",
        "reggae pop",
        "reggae rock",
        "rocksteady",
        "roots reggae",
        "ska",
        "soca",
    },

    "Afrobeats/African Pop": {
        "afro adura",
        "afro r&b",
        "afro soul",
        "afrobeat",
        "afrobeats",
        "afrogospel",
        "afropiano",
        "afropop",
        "afroswing",
        "alté",
        "amapiano",
        "asakaa",
        "azonto",
        "bongo flava",
        "ghanaian hip hop",
        "highlife",
        "hiplife",
        "nigerian drill",
        "rumba congolaise",
    },

    "Latin/Tropical": {
        "bachata",
        "bolero",
        "bossa nova",
        "brazilian funk",
        "chilean mambo",
        "christian reggaeton",
        "corridos bélicos",
        "corridos tumbados",
        "cumbia",
        "cumbia norteña",
        "cumbia sonidera",
        "electro corridos",
        "electrocumbia",
        "forró",
        "forró tradicional",
        "funk carioca",
        "latin",
        "latin afrobeats",
        "latin alternative",
        "latin country",
        "latin dance",
        "latin folk",
        "latin folklore",
        "latin hip hop",
        "latin house",
        "latin indie",
        "latin jazz",
        "latin pop",
        "latin r&b",
        "latin rock",
        "mambo",
        "mariachi",
        "merengue",
        "mpb",
        "música mexicana",
        "new mpb",
        "norteño",
        "norteño-sax",
        "pagode",
        "pagode baiano",
        "ranchera",
        "reggaeton",
        "reggaeton chileno",
        "reggaeton mexa",
        "salsa",
        "samba",
        "sertanejo",
        "sertanejo tradicional",
        "sertanejo universitário",
        "tango",
        "trap latino",
        "urbano latino",
    },

    "Indie": {
        "indie",
        "chinese indie",
        "german indie",
        "hindi indie",
        "indian indie",
        "indie québécois",
        "indonesian indie",
        "japanese indie",
        "lo-fi indie",
        "mexican indie",
        "pinoy indie",
        "taiwanese indie",
        "tamil indie",
        "vietnam indie",
    },

    "R&B/Soul": {
        "alternative r&b",
        "classic soul",
        "dark r&b",
        "funk",
        "indie r&b",
        "indie soul",
        "motown",
        "neo soul",
        "new jack swing",
        "northern soul",
        "philly soul",
        "quiet storm",
        "r&b",
        "retro soul",
        "soul",
        "trap soul",
    },

    "Hip-Hop": {
        "alternative hip hop",
        "boom bap",
        "cloud rap",
        "crunk",
        "drill",
        "east coast hip hop",
        "emo rap",
        "experimental hip hop",
        "g-funk",
        "gangster rap",
        "hardcore hip hop",
        "hip hop",
        "melodic rap",
        "old school hip hop",
        "rap",
        "southern hip hop",
        "trap",
        "underground hip hop",
        "west coast hip hop",
    },

    "Electronic/Dance": {
        "deep house",
        "disco",
        "drum and bass",
        "dubstep",
        "ebm",
        "edm",
        "electro house",
        "electronic",
        "eurodance",
        "future bass",
        "hardstyle",
        "house",
        "melodic bass",
        "progressive house",
        "synthpop",
        "tech house",
        "techno",
        "trance",
        "tropical house",
    },

    "Country": {
        "alt country",
        "americana",
        "bluegrass",
        "classic country",
        "country",
        "honky tonk",
        "outlaw country",
        "red dirt",
        "texas country",
        "traditional country",
    },

    "Jazz": {
        "bebop",
        "big band",
        "cool jazz",
        "jazz",
        "jazz funk",
        "jazz fusion",
        "smooth jazz",
        "soul jazz",
        "swing music",
        "vocal jazz",
    },

    "Blues": {
        "blues",
        "classic blues",
        "country blues",
        "delta blues",
        "modern blues",
    },

    "Classical/Musical": {
        "baroque",
        "chamber music",
        "choral",
        "classical",
        "classical crossover",
        "contemporary classical",
        "musicals",
        "opera",
        "orchestral",
        "renaissance",
    },

    "Folk": {
        "anti-folk",
        "celtic folk",
        "folk",
        "folk pop",
        "indie folk",
        "singer-songwriter",
        "traditional folk",
    },

    "Rock": {
        "alternative rock",
        "art rock",
        "blues rock",
        "classic rock",
        "death metal",
        "funk rock",
        "garage rock",
        "glam rock",
        "grunge",
        "hard rock",
        "hardcore",
        "indie rock",
        "industrial metal",
        "industrial rock",
        "metal",
        "metalcore",
        "pop punk",
        "post-grunge",
        "post-hardcore",
        "power metal",
        "progressive rock",
        "psychedelic rock",
        "psychobilly",
        "punk",
        "rock",
        "rock and roll",
        "ska punk",
        "skate punk",
        "soft rock",
        "yacht rock",
    },

    "Pop": {
        "art pop",
        "baroque pop",
        "bedroom pop",
        "dance pop",
        "electropop",
        "funk pop",
        "indie pop",
        "j-pop",
        "k-pop",
        "pop",
        "power pop",
        "soft pop",
    },
}

# Additional explicit genre tags discovered during audit

GENRE_VOTE_TAGS["Rock"].update({
    "christian rock",
    "christian alternative rock",
    "southern rock",
    "celtic rock",
    "j-rock",
    "krautrock",
})

GENRE_VOTE_TAGS["Country"].update({
    "country christian",
    "christian bluegrass",
    "southern gospel",
})

GENRE_VOTE_TAGS["Folk"].update({
    "christian folk",
    "celtic",
})

GENRE_VOTE_TAGS["Pop"].update({
    "christian pop",
    "pop worship",
    "dream pop",
    "acoustic pop",
})

GENRE_VOTE_TAGS["R&B/Soul"].update({
    "christian r&b",
    "uk r&b",
})

GENRE_VOTE_TAGS["Hip-Hop"].update({
    "christian hip hop",
    "grime",
    "uk grime",
    "chinese hip hop",
    "k-rap",
    "miami bass",
})

GENRE_VOTE_TAGS["Electronic/Dance"].update({
    "alternative dance",
    "electro swing",
    "synthwave",
    "witch house",
})

GENRE_VOTE_TAGS["Jazz"].update({
    "acid jazz",
    "experimental jazz",
})

GENRE_VOTE_TAGS["Classical/Musical"].update({
    "gregorian chant",
})

# Additional religious subgenre votes that still preserve the separate
# religious_music flag.

GENRE_VOTE_TAGS["Jazz"].update({
    "christian jazz",
})

GENRE_VOTE_TAGS["R&B/Soul"].update({
    "funk gospel",
})

GENRE_VOTE_TAGS["Latin/Tropical"].update({
    "brazilian gospel",
    "pagode gospel",
    "sertanejo gospel",
})

# Tags that are meaningful but too cross-genre to force a primary classification.

INHERIT_TAGS = {
    "industrial",
    "darkwave",
    "new wave",
    "riddim",
    "folk rock",
    "country rock",
    "gospel",
    "traditional gospel",
}

# Descriptors/context tags that should not vote for a primary musical genre.

PRIMARY_GENRE_IGNORE_TAGS = {
    "christian",
    "christmas",
    "worship",
    "ccm",
    "hymns",
    "soundtrack",
    "children's music",
    "comedy",
}

### Sanity check: a niche tag should not vote for multiple categories

This catches accidental duplicates in the manual vote sets before classification runs.

In [10]:
tag_to_categories = {}

for category, tags in GENRE_VOTE_TAGS.items():
    for tag in tags:
        tag_to_categories.setdefault(tag, []).append(category)

duplicate_vote_tags = {
    tag: categories
    for tag, categories in tag_to_categories.items()
    if len(categories) > 1
}

assert not duplicate_vote_tags, (
    "Some niche tags vote for multiple categories: "
    f"{duplicate_vote_tags}"
)

print(
    "Unique high-confidence voting tags:",
    len(tag_to_categories),
)

Unique high-confidence voting tags: 273


In [11]:
vote_tags = set().union(*GENRE_VOTE_TAGS.values())

rule_conflicts = {
    "vote_and_inherit": vote_tags & INHERIT_TAGS,
    "vote_and_ignore": vote_tags & PRIMARY_GENRE_IGNORE_TAGS,
    "inherit_and_ignore": INHERIT_TAGS & PRIMARY_GENRE_IGNORE_TAGS,
}

rule_conflicts = {
    name: tags
    for name, tags in rule_conflicts.items()
    if tags
}

assert not rule_conflicts, (
    f"Conflicting genre rules found: {rule_conflicts}"
)

print("No conflicting genre rules.")

No conflicting genre rules.


## 8. Score niche tags and apply primary-genre overrides

Every exact tag match contributes one vote to its broad category.

Why exact matching? Substring matching causes false positives such as:

- `ska` matching `skate punk`
- `dub` matching `dubstep`
- `rap` matching `trap`

If multiple categories tie for the highest score, the notebook keeps the mapped official genre rather than forcing a questionable override.

In [12]:
def genre_scores(tags):
    tag_set = {
        str(tag).lower().strip()
        for tag in tags
        if pd.notna(tag)
    }

    return {
        category: len(tag_set & category_tags)
        for category, category_tags in GENRE_VOTE_TAGS.items()
    }


def choose_genre_override(tags):
    scores = genre_scores(tags)

    best_score = max(scores.values())

    if best_score == 0:
        return None

    winners = [
        category
        for category, score in scores.items()
        if score == best_score
    ]

    # Ambiguous niche evidence: keep official/base genre.
    if len(winners) != 1:
        return None

    return winners[0]

In [13]:
songs["genre_override"] = (
    songs["niche_genre_list"]
    .apply(choose_genre_override)
)

songs["genre_engineered"] = (
    songs["genre_override"]
    .fillna(songs["genre_base"])
)

unexpected_genres = (
    set(songs["genre_engineered"].dropna().unique())
    - set(FINAL_GENRES)
)

assert not unexpected_genres, (
    f"Unexpected engineered genres: {unexpected_genres}"
)

## 8A. Create secondary thematic features

Some niche tags describe themes, communities, or contexts rather than a
primary musical genre.

Religious tags therefore do not determine `genre_engineered`, but are
preserved as a separate binary feature.

This allows a song to be, for example:

- `genre_engineered = "Hip-Hop"`
- `religious_music = 1`

rather than forcing "Christian" to become a primary genre.

In [14]:
RELIGIOUS_TAGS = {
    "christian",
    "christian alternative rock",
    "christian bluegrass",
    "christian folk",
    "christian hip hop",
    "christian jazz",
    "christian lo-fi",
    "christian pop",
    "christian r&b",
    "christian reggae",
    "christian reggaeton",
    "christian rock",
    "country christian",
    "ccm",
    "gospel",
    "traditional gospel",
    "southern gospel",
    "afrogospel",
    "brazilian gospel",
    "funk gospel",
    "pagode gospel",
    "sertanejo gospel",
    "worship",
    "pop worship",
    "hymns",
    "catholic music"
}

In [15]:
def has_religious_tag(tags):
    tag_set = {
        str(tag).lower().strip()
        for tag in tags
        if pd.notna(tag)
    }

    return int(bool(tag_set & RELIGIOUS_TAGS))


songs["religious_music"] = (
    songs["niche_genre_list"]
    .apply(has_religious_tag)
)

In [16]:
print(
    songs["religious_music"]
    .value_counts(dropna=False)
)

print(
    f"Religiously tagged songs: "
    f"{songs['religious_music'].sum():,}"
)

religious_music
0    520298
1     30324
Name: count, dtype: int64
Religiously tagged songs: 30,324


In [17]:
display(
    songs.loc[
        songs["religious_music"] == 1,
        [
            "name",
            "artists",
            "genre_original",
            "genre_engineered",
            "niche_genres",
        ]
    ].sample(30, random_state=42)
)

,name,artists,genre_original,genre_engineered,niche_genres
241320,It Is Well (Live),"[""Bethel Music"", ""Kristene DiMarco""]",Pop,Pop,"[""ccm"", ""christian"", ""pop worship"", ""worship""]"
212337,I Fade Away,"[""Pillar""]",Rock,Rock,"[""chinese rock"", ""christian alternative rock"",..."
72204,Burning Bridges,"[""Seventh Day Slumber""]",Rock,Rock,"[""ccm"", ""chinese rock"", ""christian"", ""christia..."
147223,Farther Along (Live),"[""The Petersens""]",Country,Country,"[""bluegrass"", ""christian bluegrass"", ""christia..."
176039,Gone Country,"[""Alan Jackson""]",Country,Country,"[""classic country"", ""country"", ""country christ..."
180083,Gravity,"[""John Schlitt""]",Rock,Rock,"[""ccm"", ""christian alternative rock"", ""christi..."
199713,Holding Onto Hope,"[""I AM THEY""]",Folk,Folk,"[""ccm"", ""christian"", ""christian folk"", ""worship""]"
218664,I Then Shall Live - A Cappella,"[""Gaither Vocal Band""]",Country,Country,"[""christian"", ""christian bluegrass"", ""country ..."
389729,Searching - Stripped Version,"[""Jlyricz""]",Pop,Afrobeats/African Pop,"[""afrogospel"", ""afropop""]"
403627,Sinking Stone,"[""Alison Krauss & Union Station""]",Country,Country,"[""bluegrass"", ""christian bluegrass"", ""newgrass""]"


## 9. Audit the engineered result

Keep an audit trail rather than replacing the source columns.

Useful columns:

- `genre_original`: source dataset's broad genre
- `genre_base`: source genre renamed into the final taxonomy
- `niche_genres`: original niche-tag string
- `genre_override`: niche-tag winner, if there was one
- `genre_engineered`: final category used for analysis

In [18]:
genre_summary = (
    songs["genre_engineered"]
    .value_counts(dropna=False)
    .rename_axis("genre_engineered")
    .reset_index(name="song_count")
)

display(genre_summary)

,genre_engineered,song_count
0,Rock,207008
1,Electronic/Dance,67840
2,Pop,58224
3,Country,46178
4,Hip-Hop,40467
5,R&B/Soul,32787
6,Folk,31818
7,Jazz,17846
8,Reggae/Dancehall,15770
9,Blues,11923


In [19]:
changed_genres = songs.loc[
    songs["genre_engineered"] != songs["genre_base"],
    [
        "name",
        "artists",
        "genre_original",
        "genre_base",
        "niche_genres",
        "genre_override",
        "genre_engineered",
    ],
]

print(f"Rows changed by niche-tag overrides: {len(changed_genres):,}")

if len(changed_genres) > 0:
    display(
        changed_genres.sample(
            min(40, len(changed_genres)),
            random_state=42,
        )
    )

Rows changed by niche-tag overrides: 69,518


,name,artists,genre_original,genre_base,niche_genres,genre_override,genre_engineered
480102,Throne,"[""Original Koffee""]",Folk,Folk,"[""afrobeats"", ""dancehall"", ""reggae""]",Reggae/Dancehall,Reggae/Dancehall
49014,Bebe,"[""2Baba"", ""Larry Gaaga"", ""Mi Casa""]",R&B,R&B/Soul,"[""afro house"", ""afro r&b"", ""afrobeat"", ""afrobe...",Afrobeats/African Pop,Afrobeats/African Pop
175513,Gold to Me,"[""Dirty Heads""]",Folk,Folk,"[""reggae"", ""reggae rock""]",Reggae/Dancehall,Reggae/Dancehall
178798,Got Me,"[""Roy Woods""]",Hip-Hop,Hip-Hop,"[""trap soul""]",R&B/Soul,R&B/Soul
425169,Still Pt. 3,"[""A-One"", ""K Koke""]",Electronic,Electronic/Dance,"[""grime"", ""uk grime""]",Hip-Hop,Hip-Hop
298272,Miley Cyrus (Twerk),"[""Elephant Man""]",Folk,Folk,"[""dancehall"", ""ragga"", ""reggae"", ""riddim"", ""so...",Reggae/Dancehall,Reggae/Dancehall
318028,New Mess,"[""Helvetia""]",Rock,Rock,"[""lo-fi indie"", ""shoegaze"", ""slowcore""]",Indie,Indie
507145,Walking in the Air,"[""Howard Blake"", ""Laura Auer"", ""Broadway Studi...",Pop,Pop,"[""christmas"", ""classical crossover""]",Classical/Musical,Classical/Musical
249138,Jumbo,"[""Ice Prince""]",R&B,R&B/Soul,"[""afrobeat"", ""afrobeats"", ""afropop"", ""azonto"",...",Afrobeats/African Pop,Afrobeats/African Pop
508761,Wash Away,"[""Stone Senses""]",Folk,Folk,"[""reggae"", ""reggae rock""]",Reggae/Dancehall,Reggae/Dancehall


## 10. Tag-rule coverage audit

There are 754 unique niche tags in this dataset, but they do **not** all need a manual category.

This audit distinguishes:

- `vote` — explicitly supplies evidence for a category
- `inherit` — deliberately cross-genre; use the official/base genre unless other tags resolve it
- `ignore` — thematic/contextual; does not vote
- `fallback_to_official` — not manually mapped because the base broad genre remains the default

### `vote`

- This niche tag actively contributes a vote toward a primary genre.
- `manual_vote_category` is populated.

### `inherit`

- Deliberately cross-genre.
- This tag itself does not vote.
- Another tag on the same song can still cause an override.
- Otherwise, keep `genre_base`.

### `ignore`

- Deliberately not a primary-genre descriptor.
- Examples: `christian`, `christmas`, `worship`.
- Does not vote, but can still be used for a separate feature.

### `fallback_to_official`

- We have not assigned this niche tag a manual rule.
- It does not vote.
- If no other tag votes, keep `genre_base`.

This means every niche tag is covered by the **classification logic**, even though only a subset requires an explicit override.

In [20]:
all_niche_tags = set(
    songs["niche_genre_list"]
    .explode()
    .dropna()
    .astype(str)
    .str.lower()
    .str.strip()
)

vote_tags = set(tag_to_categories)

def get_rule_type(tag):
    if tag in vote_tags:
        return "vote"
    if tag in INHERIT_TAGS:
        return "inherit"
    if tag in PRIMARY_GENRE_IGNORE_TAGS:
        return "ignore"
    return "fallback_to_official"


tag_rule_summary = pd.Series(
    [get_rule_type(tag) for tag in all_niche_tags]
).value_counts()

print("Total niche genres:", len(all_niche_tags))
display(tag_rule_summary.rename("tag_count"))

Total niche genres: 754


fallback_to_official    467
vote                    271
ignore                    8
inherit                   8
Name: tag_count, dtype: int64

## 11. Optional: inspect the saved 754-tag audit

The previously generated `genre_tag_audit.csv` is useful for reviewing high-volume, low-agreement tags without rebuilding the file every time.

Low agreement is a **review flag**, not proof that a tag needs an override. Some tags genuinely span genres.

In [21]:
GENRE_AUDIT_PATH = DATA_PROCESSED / "genre_tag_audit.csv"

if GENRE_AUDIT_PATH.exists():
    genre_audit = pd.read_csv(GENRE_AUDIT_PATH)

    review_candidates = (
        genre_audit[
            (genre_audit["total_count"] >= 500)
            & (genre_audit["dominant_share"] < 0.80)
        ]
        .copy()
    )

    review_candidates["rule_type"] = (
        review_candidates["niche_genre"]
        .map(get_rule_type)
    )

    review_candidates["manual_vote_category"] = (
        review_candidates["niche_genre"]
        .map(
            {
                tag: category
                for category, tags in GENRE_VOTE_TAGS.items()
                for tag in tags
            }
        )
    )

    display(
        review_candidates[
            [
                "niche_genre",
                "total_count",
                "dominant_genre",
                "dominant_share",
                "second_genre",
                "second_share",
                "rule_type",
                "manual_vote_category",
                "examples",
            ]
        ].sort_values("total_count", ascending=False)
    )

else:
    print(
        "genre_tag_audit.csv not found. "
        "The rebuild code is preserved in the appendix."
    )

,niche_genre,total_count,dominant_genre,dominant_share,second_genre,second_share,rule_type,manual_vote_category,examples
8,pop punk,13001,Rock,0.598185,Pop,0.363280,vote,Rock,"!!Noble Stabbings!! — [""Dillinger Four""] | ""Co..."
14,blues rock,11499,Blues,0.788155,Rock,0.201670,vote,Rock,"""B"" Movie Box Car Blues - Live — [""The Blues B..."
16,industrial,10991,Rock,0.705395,Electronic,0.290601,inherit,NaN,"""President Dead"" — [""Marilyn Manson""] | $10 Bi..."
18,christian,10761,Pop,0.394666,Rock,0.300158,ignore,NaN,"#LETSGO - Live — [""Planetshakers""] | #LITO — [..."
20,ska punk,10346,Folk,0.520588,Rock,0.468200,vote,Rock,"#NotRealLife — [""H2O""] | $1000 Bender — [""Get ..."
...,...,...,...,...,...,...,...,...,...
362,miami bass,594,Electronic,0.629630,Hip-Hop,0.190236,vote,Hip-Hop,"2 Live Blues — [""2 LIVE CREW""] | 2 Live Is Her..."
363,soundtrack,590,Classical,0.666102,Pop,0.193220,ignore,NaN,"""DEVILS NEVER CRY""(スタッフロール) — [""Capcom Sound T..."
365,cajun,583,Blues,0.651801,Country,0.140652,fallback_to_official,NaN,"(Everybody Wanna Get Rich) Rite Away — [""Dr. J..."
369,dark ambient,556,Electronic,0.491007,Rock,0.483813,fallback_to_official,NaN,(Don’t Fear) the Reaper (Blue Oyster Cult) — [...


## 12. Save the engineered metadata

This saved file becomes the lightweight song-level lookup table for the matching stage.

The original `songs.csv` remains untouched.

In [22]:
ENGINEERED_METADATA_PATH = (
    DATA_PROCESSED / "song_metadata_engineered.parquet"
)

songs.to_parquet(
    ENGINEERED_METADATA_PATH,
    index=False,
)

print("Saved:", ENGINEERED_METADATA_PATH)

Saved: /Users/mercyeme/Documents/Data_Projects_2026/RecessionPop/data/processed/song_metadata_engineered.parquet


# Next step — Billboard ↔ 550K song matching

The genre work should now be treated as a **frozen first-pass taxonomy** unless the audit reveals a serious systematic error.

The next substantive step is matching each unique Billboard `Song` + `Artist` pair to the 550K song table, obtaining a canonical song ID, and then merging features back to the full weekly Billboard chart.

The matching code below is preserved but commented out so it does not run before you are ready.

In [23]:
# --- NEXT STEP: text normalization for exact matching ---
#
# def normalize_text(value):
#     if pd.isna(value):
#         return ""
#
#     value = str(value).lower().strip()
#     value = unicodedata.normalize("NFKD", value)
#     value = "".join(
#         char
#         for char in value
#         if not unicodedata.combining(char)
#     )
#     value = value.replace("&", "and")
#     value = re.sub(r"[^\w\s]", " ", value)
#     value = re.sub(r"\s+", " ", value)
#
#     return value.strip()
#
#
# billboard_songs["title_key"] = (
#     billboard_songs["Song"]
#     .map(normalize_text)
# )
#
# billboard_songs["artist_key"] = (
#     billboard_songs["Artist"]
#     .map(normalize_text)
# )
#
# # The 550K `artists` field is itself a stringified list.
# def parse_artist_list(value):
#     if isinstance(value, list):
#         return value
#
#     if pd.isna(value):
#         return []
#
#     try:
#         parsed = ast.literal_eval(value)
#         return parsed if isinstance(parsed, list) else []
#
#     except (ValueError, SyntaxError, TypeError):
#         return []
#
#
# songs["artist_list"] = (
#     songs["artists"]
#     .apply(parse_artist_list)
# )
#
# songs["primary_artist"] = (
#     songs["artist_list"]
#     .str[0]
# )
#
# songs["title_key"] = (
#     songs["name"]
#     .map(normalize_text)
# )
#
# songs["artist_key"] = (
#     songs["primary_artist"]
#     .map(normalize_text)
# )

In [24]:
def normalize_text(value):
    if pd.isna(value):
        return ""

    value = str(value).lower().strip()
    value = unicodedata.normalize("NFKD", value)

    value = "".join(
        char
        for char in value
        if not unicodedata.combining(char)
    )

    value = value.replace("&", "and")
    value = re.sub(r"[^\w\s]", " ", value)
    value = re.sub(r"\s+", " ", value)

    return value.strip()

In [25]:
billboard_songs["title_key"] = (
    billboard_songs["Song"]
    .map(normalize_text)
)

billboard_songs["artist_key"] = (
    billboard_songs["Artist"]
    .map(normalize_text)
)

In [26]:
display(
    billboard_songs[
        [
            "Song",
            "Artist",
            "title_key",
            "artist_key",
        ]
    ].head(10)
)

,Song,Artist,title_key,artist_key
0,Poor Little Fool,Ricky Nelson,poor little fool,ricky nelson
1,Patricia,Perez Prado And His Orchestra,patricia,perez prado and his orchestra
2,Splish Splash,Bobby Darin,splish splash,bobby darin
3,Hard Headed Woman,Elvis Presley With The Jordanaires,hard headed woman,elvis presley with the jordanaires
4,When,Kalin Twins,when,kalin twins
5,Rebel-'rouser,Duane Eddy His Twangy Guitar And The Rebels,rebel rouser,duane eddy his twangy guitar and the rebels
6,Yakety Yak,The Coasters,yakety yak,the coasters
7,My True Love,Jack Scott,my true love,jack scott
8,Willie And The Hand Jive,The Johnny Otis Show,willie and the hand jive,the johnny otis show
9,Fever,Peggy Lee,fever,peggy lee


In [27]:
def parse_artist_list(value):
    if isinstance(value, list):
        return value

    if pd.isna(value):
        return []

    try:
        parsed = ast.literal_eval(value)
        return parsed if isinstance(parsed, list) else []

    except (ValueError, SyntaxError, TypeError):
        return []

In [28]:
songs["artist_list"] = (
    songs["artists"]
    .apply(parse_artist_list)
)

songs["primary_artist"] = (
    songs["artist_list"]
    .str[0]
)

In [29]:
display(
    songs[
        [
            "name",
            "artists",
            "artist_list",
            "primary_artist",
        ]
    ].head(10)
)

,name,artists,artist_list,primary_artist
0,!,"[""HELLYEAH""]",[HELLYEAH],HELLYEAH
1,!!,"[""Yxngxr1""]",[Yxngxr1],Yxngxr1
2,!!Noble Stabbings!!,"[""Dillinger Four""]",[Dillinger Four],Dillinger Four
3,!I'll Be Back!,"[""Ril\u00e8s""]",[Rilès],Rilès
4,!Lost!,"[""Ril\u00e8s""]",[Rilès],Rilès
5,!Que Vida! - Mono Version,"[""Love""]",[Love],Love
6,!liaF cipE,"[""One Morning Left""]",[One Morning Left],One Morning Left
7,""" ""","[""Breadwinner""]",[Breadwinner],Breadwinner
8,""" R.I.P ""","[""Ram-Zet""]",[Ram-Zet],Ram-Zet
9,"""...Because All Suffering Is Sweet to Me...""","[""Envy On The Coast""]",[Envy On The Coast],Envy On The Coast


In [30]:
songs["title_key"] = (
    songs["name"]
    .map(normalize_text)
)

songs["artist_key"] = (
    songs["primary_artist"]
    .map(normalize_text)
)

In [31]:
display(
    songs[
        [
            "name",
            "primary_artist",
            "title_key",
            "artist_key",
        ]
    ].head(10)
)

,name,primary_artist,title_key,artist_key
0,!,HELLYEAH,,hellyeah
1,!!,Yxngxr1,,yxngxr1
2,!!Noble Stabbings!!,Dillinger Four,noble stabbings,dillinger four
3,!I'll Be Back!,Rilès,i ll be back,riles
4,!Lost!,Rilès,lost,riles
5,!Que Vida! - Mono Version,Love,que vida mono version,love
6,!liaF cipE,One Morning Left,liaf cipe,one morning left
7,""" """,Breadwinner,,breadwinner
8,""" R.I.P """,Ram-Zet,r i p,ram zet
9,"""...Because All Suffering Is Sweet to Me...""",Envy On The Coast,because all suffering is sweet to me,envy on the coast


In [32]:
spotify_key_counts = (
    songs.loc[
        songs["title_key"].ne("")
        & songs["artist_key"].ne("")
    ]
    .groupby(
        ["title_key", "artist_key"]
    )
    .size()
    .reset_index(name="candidate_count")
)

In [33]:
display(
    spotify_key_counts.loc[
        spotify_key_counts["candidate_count"] > 1
    ]
    .sort_values(
        "candidate_count",
        ascending=False
    )
    .head(30)
)

,title_key,artist_key,candidate_count
108996,drive,the cars,70
323039,rockin around the christmas tree,brenda lee,45
232794,little saint nick 1991 remix,the beach boys,42
326033,run rudolph run,chuck berry,41
139229,frosty the snowman,ella fitzgerald,38
347827,sleigh ride,ella fitzgerald,33
190312,i saw mommy kissing santa claus,the jackson 5,29
433279,waiting for a girl like you,foreigner,28
421383,true,spandau ballet,28
225694,let it snow let it snow let it snow,dean martin,27


In [34]:
print(
    "Blank Spotify title keys:",
    songs["title_key"].eq("").sum()
)

print(
    "Blank Spotify artist keys:",
    songs["artist_key"].eq("").sum()
)

print(
    "Blank Billboard title keys:",
    billboard_songs["title_key"].eq("").sum()
)

print(
    "Blank Billboard artist keys:",
    billboard_songs["artist_key"].eq("").sum()
)

Blank Spotify title keys: 38
Blank Spotify artist keys: 72
Blank Billboard title keys: 0
Blank Billboard artist keys: 0


In [35]:
drive_duplicates = songs.loc[
    (songs["title_key"] == "drive")
    & (songs["artist_key"] == "the cars"),
    [
        "name",
        "album_name",
        "popularity",
        "danceability",
        "energy",
        "valence",
        "tempo",
        "duration_ms",
        "id",
    ]
]

display(drive_duplicates)

,name,album_name,popularity,danceability,energy,valence,tempo,duration_ms,id
128174,Drive,Power Ballads,0,0.582,0.606,0.504,111.441,234493,00lPppvNMTDI6YyS0fMJRO
128175,Drive,Greatest Love Songs,4,0.583,0.690,0.458,111.344,234320,01mG9uNxQanlBDwOs0pPQ9
128177,Drive,Chilled 80s,4,0.582,0.606,0.504,111.441,234493,03dzKrMePNq35X4zkNLA84
128179,Drive,80s Rock Ballad Classics,0,0.582,0.606,0.504,111.441,234493,0M32XE0ZnApxPz3pCD7VaS
128180,Drive,1984,0,0.582,0.606,0.504,111.441,234493,0VlTSIU8nzMlg3cjgXcScI
...,...,...,...,...,...,...,...,...,...
128289,Drive,Absolute Cinema,0,0.582,0.606,0.504,111.441,234493,71GgS1rEWz1kq7QHeGX2r0
128291,Drive,Adult Contemporary,0,0.346,0.691,0.433,111.348,235187,73jdxoZmJB0WkTsLbknplJ
128292,Drive,Play - Ballads,1,0.583,0.690,0.458,111.344,234320,77ZSLDce9F592snV4jZxp2
128293,Drive,American Radio Classics,0,0.346,0.691,0.433,111.348,235187,7AvpYVlNQeztzq0vPb54CP


In [36]:
drive_profiles = (
    drive_duplicates
    .groupby(
        [
            "danceability",
            "energy",
            "valence",
            "tempo",
            "duration_ms",
        ],
        dropna=False
    )
    .size()
    .reset_index(name="count")
    .sort_values(
        "count",
        ascending=False
    )
)

display(drive_profiles)

,danceability,energy,valence,tempo,duration_ms,count
4,0.582,0.606,0.504,111.441,234493,37
0,0.346,0.691,0.433,111.348,235187,15
5,0.583,0.690,0.458,111.344,234320,15
1,0.424,0.860,0.380,111.497,234400,1
2,0.579,0.608,0.499,111.441,234493,1
3,0.579,0.646,0.540,111.081,237600,1


In [37]:
dominant_drive = drive_duplicates.loc[
    (drive_duplicates["danceability"] == 0.582)
    & (drive_duplicates["energy"] == 0.606)
    & (drive_duplicates["valence"] == 0.504)
    & (drive_duplicates["tempo"] == 111.441)
    & (drive_duplicates["duration_ms"] == 234493)
]

display(
    dominant_drive[
        [
            "album_name",
            "popularity",
            "id",
        ]
    ]
    .sort_values(
        "popularity",
        ascending=False
    )
)

,album_name,popularity,id
128240,Heartbeat City,68,3wfujdbamR3Z46F4xav7LM
128204,The Greatest 80's Hits,5,2LgO6KzZQX780PTWOo0V2D
128177,Chilled 80s,4,03dzKrMePNq35X4zkNLA84
128202,Oldies - Ballads,2,2IyYDi02VuoQPhtRV3w5iS
128269,Greatest 80s Love Songs,1,5o59KQnhvKV3Z9xsIrmzlh
128220,Sound of the 80's,1,38vGxEPXVsLXiexd08F40w
128280,80s Memories,1,6i6Vp9TJImsZBQQ12IFfVP
128276,80's Electronic Mixtape,0,6MWs4heHuYQrMoi1zEb4UF
128274,30 Great Ballads,0,6CdpYcsyJId9HEz2dvMmkm
128273,Love and Rock,0,68fh5kX0nEDDNATR7l3HWw


In [38]:
last_last_duplicates = songs.loc[
    (songs["title_key"] == "last last")
    & (songs["artist_key"] == "burna boy"),
    [
        "name",
        "album_name",
        "popularity",
        "danceability",
        "energy",
        "valence",
        "tempo",
        "duration_ms",
        "id",
    ]
]

display(last_last_duplicates)

,name,album_name,popularity,danceability,energy,valence,tempo,duration_ms,id
259922,Last Last,Daily Pop Mix,0,0.795,0.565,0.55,87.925,172342,03um6hfYLzq0p4c3Lk0Dld
259923,Last Last,Internet Sensations,0,0.795,0.565,0.55,87.925,172342,0hWBiQZZ7btfzKoMDUUHCH
259924,Last Last,Pop Brandneu,0,0.795,0.565,0.55,87.925,172342,0ozGUAz5IE3DAWoe7C1kG2
259925,Last Last,Hip Hop & R'n'B,0,0.795,0.565,0.55,87.925,172342,0t1CW2tjQq45DEdo7thJo1
259926,Last Last,Comme un lundi,0,0.795,0.565,0.55,87.925,172342,13lcPEauPnz6ynDJIfHZAR
259927,Last Last,Totally Rap,0,0.795,0.565,0.55,87.925,172342,1NdGOt0hJikfIy2OUkXdi2
259928,Last Last,2Step - Pop R'n'B,0,0.795,0.565,0.55,87.925,172342,27C9LOKwqrjYNHwZ5pu8zH
259929,Last Last,Summer Music Festival Hits,0,0.795,0.565,0.55,87.925,172342,2ECokGI5VuABjcNb90be3p
259930,Last Last,Last Last,0,0.795,0.565,0.55,87.925,172342,2PDgArI0p7UkeYgXWuqpmh
259931,Last Last,20's R'n'B,0,0.795,0.565,0.55,87.925,172342,2cFjQQ33h8NiQAKqpJGhgd


In [39]:
last_last_profiles = (
    last_last_duplicates
    .groupby(
        [
            "danceability",
            "energy",
            "valence",
            "tempo",
            "duration_ms",
        ],
        dropna=False
    )
    .size()
    .reset_index(name="count")
    .sort_values(
        "count",
        ascending=False
    )
)

display(last_last_profiles)

,danceability,energy,valence,tempo,duration_ms,count
0,0.795,0.565,0.55,87.925,172342,21
1,0.812,0.563,0.54,87.979,172342,1


In [40]:
match_cols = [
    "id",
    "name",
    "artists",
    "primary_artist",
    "title_key",
    "artist_key",
    "year",
    "popularity",
    "valence",
    "energy",
    "danceability",
    "tempo",
    "mode",
    "genre_original",
    "genre_engineered",
    "religious_music",
]

In [41]:
spotify_lookup = (
    songs[
        match_cols
        + [
            "duration_ms",
            "album_name",
        ]
    ]
    .copy()
)

In [42]:
display(
    spotify_lookup[
        [
            "name",
            "primary_artist",
            "album_name",
            "duration_ms",
            "popularity",
        ]
    ].head()
)

,name,primary_artist,album_name,duration_ms,popularity
0,!,HELLYEAH,UNDEN!ABLE,79500,0
1,!!,Yxngxr1,Childhood Dreams,114000,29
2,!!Noble Stabbings!!,Dillinger Four,Situationist Comedy,197400,0
3,!I'll Be Back!,Rilès,!I'll Be Back!,178533,43
4,!Lost!,Rilès,!Lost!,186197,0


In [43]:
spotify_lookup["sig_danceability"] = (
    spotify_lookup["danceability"]
    .round(3)
)

spotify_lookup["sig_energy"] = (
    spotify_lookup["energy"]
    .round(3)
)

spotify_lookup["sig_valence"] = (
    spotify_lookup["valence"]
    .round(3)
)

spotify_lookup["sig_tempo"] = (
    spotify_lookup["tempo"]
    .round(1)
)

spotify_lookup["sig_duration"] = (
    spotify_lookup["duration_ms"]
    .div(1000)
    .round()
)

In [44]:
display(
    spotify_lookup[
        [
            "name",
            "danceability",
            "sig_danceability",
            "energy",
            "sig_energy",
            "valence",
            "sig_valence",
            "tempo",
            "sig_tempo",
            "duration_ms",
            "sig_duration",
        ]
    ].head(10)
)

,name,danceability,sig_danceability,energy,sig_energy,valence,sig_valence,tempo,sig_tempo,duration_ms,sig_duration
0,!,0.415,0.415,0.605,0.605,0.193,0.193,100.059,100.1,79500,80.0
1,!!,0.788,0.788,0.648,0.648,0.287,0.287,79.998,80.0,114000,114.0
2,!!Noble Stabbings!!,0.171,0.171,0.957,0.957,0.349,0.349,175.317,175.3,197400,197.0
3,!I'll Be Back!,0.823,0.823,0.612,0.612,0.688,0.688,142.959,143.0,178533,179.0
4,!Lost!,0.729,0.729,0.552,0.552,0.380,0.380,86.103,86.1,186197,186.0
5,!Que Vida! - Mono Version,0.600,0.600,0.540,0.540,0.547,0.547,125.898,125.9,220560,221.0
6,!liaF cipE,0.483,0.483,0.894,0.894,0.284,0.284,140.058,140.1,200200,200.0
7,""" """,0.296,0.296,0.831,0.831,0.217,0.217,124.921,124.9,201760,202.0
8,""" R.I.P """,0.474,0.474,0.923,0.923,0.288,0.288,159.974,160.0,533160,533.0
9,"""...Because All Suffering Is Sweet to Me...""",0.241,0.241,0.734,0.734,0.040,0.040,157.355,157.4,237973,238.0


In [45]:
signature_cols = [
    "sig_danceability",
    "sig_energy",
    "sig_valence",
    "sig_tempo",
    "sig_duration",
]

In [46]:
signature_counts = (
    spotify_lookup.loc[
        spotify_lookup["title_key"].ne("")
        & spotify_lookup["artist_key"].ne("")
    ]
    .groupby(
        [
            "title_key",
            "artist_key",
        ]
        + signature_cols,
        dropna=False,
    )
    .size()
    .reset_index(name="signature_count")
)

In [47]:
display(
    signature_counts.loc[
        (
            (signature_counts["title_key"] == "drive")
            & (signature_counts["artist_key"] == "the cars")
        )
        |
        (
            (signature_counts["title_key"] == "last last")
            & (signature_counts["artist_key"] == "burna boy")
        )
    ]
    .sort_values(
        [
            "title_key",
            "signature_count",
        ],
        ascending=[
            True,
            False,
        ],
    )
)

,title_key,artist_key,sig_danceability,sig_energy,sig_valence,sig_tempo,sig_duration,signature_count
121524,drive,the cars,0.582,0.606,0.504,111.4,234.0,37
121520,drive,the cars,0.346,0.691,0.433,111.3,235.0,15
121525,drive,the cars,0.583,0.690,0.458,111.3,234.0,15
121521,drive,the cars,0.424,0.860,0.380,111.5,234.0,1
121522,drive,the cars,0.579,0.608,0.499,111.4,234.0,1
121523,drive,the cars,0.579,0.646,0.540,111.1,238.0,1
248417,last last,burna boy,0.795,0.565,0.550,87.9,172.0,21
248418,last last,burna boy,0.812,0.563,0.540,88.0,172.0,1


In [48]:
spotify_lookup = (
    spotify_lookup
    .merge(
        signature_counts,
        on=[
            "title_key",
            "artist_key",
        ] + signature_cols,
        how="left",
    )
)

In [49]:
display(
    spotify_lookup.loc[
        (
            (spotify_lookup["title_key"] == "drive")
            & (spotify_lookup["artist_key"] == "the cars")
        )
        |
        (
            (spotify_lookup["title_key"] == "last last")
            & (spotify_lookup["artist_key"] == "burna boy")
        ),
        [
            "title_key",
            "name",
            "primary_artist",
            "album_name",
            "popularity",
            "signature_count",
        ]
    ]
    .sort_values(
        [
            "title_key",
            "signature_count",
            "popularity",
        ],
        ascending=[
            True,
            False,
            False,
        ],
    )
)

,title_key,name,primary_artist,album_name,popularity,signature_count
128240,drive,Drive,The Cars,Heartbeat City,68,37.0
128204,drive,Drive,The Cars,The Greatest 80's Hits,5,37.0
128177,drive,Drive,The Cars,Chilled 80s,4,37.0
128202,drive,Drive,The Cars,Oldies - Ballads,2,37.0
128220,drive,Drive,The Cars,Sound of the 80's,1,37.0
...,...,...,...,...,...,...
259939,last last,Last Last,Burna Boy,New Pop Classics,0,21.0
259940,last last,Last Last,Burna Boy,African Heat - Afro Hits,0,21.0
259941,last last,Last Last,Burna Boy,Good Vibes,0,21.0
259943,last last,Last Last,Burna Boy,Pop Hits Now,0,21.0


In [50]:
spotify_exact_candidates = (
    spotify_lookup.loc[
        spotify_lookup["title_key"].ne("")
        & spotify_lookup["artist_key"].ne("")
    ]
    .copy()
)

In [51]:
spotify_exact_canonical = (
    spotify_exact_candidates
    .sort_values(
        [
            "title_key",
            "artist_key",
            "signature_count",
            "popularity",
            "id",
        ],
        ascending=[
            True,
            True,
            False,
            False,
            True,
        ],
    )
    .drop_duplicates(
        subset=[
            "title_key",
            "artist_key",
        ],
        keep="first",
    )
    .reset_index(drop=True)
)

In [59]:
display(
    spotify_exact_canonical.loc[
        spotify_exact_canonical["title_key"].isin(
            ["drive", "last last"]
        ),
        [
            "title_key",
            "name",
            "primary_artist",
            "album_name",
            "popularity",
            "signature_count",
            "id",
        ]
    ]
)

,title_key,name,primary_artist,album_name,popularity,signature_count,id
108957,drive,Drive,Adana Twins,Drive,24,1.0,4a6Lbm2WkVTKhzJX7RtJXq
108958,drive,Drive,Annihilator,Schizo Deluxe,13,1.0,11Z116fi85akAp25zZtFaD
108959,drive,Drive,Ashley Monroe,Drive,18,1.0,6is4HZ5UFpIPGxUNdVdKbV
108960,drive,Drive,Assemblage 23,Defiance,24,1.0,7D5SH6KSuQ4hepj3sy9vaK
108961,drive,Drive,Black Coffee,Drive,0,1.0,67aiZtmB5ZBCGxBN3pOL6C
108962,drive,Drive,Black Stone Cherry,Black Stone Cherry,28,1.0,6jUzf4VCRXmR8KWNvQznIe
108963,drive,Drive,Blank & Jones,Drive,20,1.0,4zR1K26LJZ2ZqDOGtCyZRs
108964,drive,Drive,Boyce Avenue,"Cover Sessions, Vol. 3",37,1.0,6vcZ52pQeAJMLrcLyc0X4O
108965,drive,Drive,Cavo,Drive,9,1.0,50Rh7kUdpSNUBQZz7sAbtZ
108966,drive,Drive,Charon,The Dying Daylights,18,1.0,46LxIpygWs0y3DxTJuwphv


In [60]:
display(
    spotify_exact_canonical.loc[
        (
            (spotify_exact_canonical["title_key"] == "drive")
            & (spotify_exact_canonical["artist_key"] == "the cars")
        )
        |
        (
            (spotify_exact_canonical["title_key"] == "last last")
            & (spotify_exact_canonical["artist_key"] == "burna boy")
        ),
        [
            "title_key",
            "artist_key",
            "name",
            "primary_artist",
            "album_name",
            "popularity",
            "signature_count",
            "id",
        ]
    ]
)

,title_key,artist_key,name,primary_artist,album_name,popularity,signature_count,id
108996,drive,the cars,Drive,The Cars,Heartbeat City,68,37.0,3wfujdbamR3Z46F4xav7LM
221706,last last,burna boy,Last Last,Burna Boy,"Love, Damini",68,21.0,5YbPxJwPfrj7uswNwoF1pJ


In [61]:
exact_matches = (
    billboard_songs
    .merge(
        spotify_exact_canonical,
        on=[
            "title_key",
            "artist_key",
        ],
        how="left",
        indicator=True,
        suffixes=(
            "_billboard",
            "_spotify",
        ),
    )
)

In [62]:
print(
    exact_matches["_merge"]
    .value_counts()
)

_merge
left_only     22126
both          10466
right_only        0
Name: count, dtype: int64


In [63]:
unmatched = (
    exact_matches.loc[
        exact_matches["_merge"] == "left_only",
        [
            "Song",
            "Artist",
            "title_key",
            "artist_key",
        ]
    ]
    .copy()
)

In [64]:
display(
    unmatched.sample(
        30,
        random_state=42
    )
)

,Song,Artist,title_key,artist_key
5607,Pandora's Golden Heebie Jeebies,The Association,pandora s golden heebie jeebies,the association
29130,Beautiful Trip,Kid Cudi,beautiful trip,kid cudi
22461,Amor Eterno,Cristian Castro,amor eterno,cristian castro
4672,"""Oowee, Oowee""",Perry Como,oowee oowee,perry como
17098,Blame It On The Rain,Milli Vanilli,blame it on the rain,milli vanilli
29597,Scoop,Lil Nas X Featuring Doja Cat,scoop,lil nas x featuring doja cat
22599,Do It To It,Cherish Featuring Sean Paul Of The YoungBloodZ,do it to it,cherish featuring sean paul of the youngbloodz
16395,Throwaway,Mick Jagger,throwaway,mick jagger
10702,Swing Your Daddy,Jim Gilstrap,swing your daddy,jim gilstrap
2426,Oh My Angel,Bertha Tillman,oh my angel,bertha tillman


In [65]:
unmatched_title_check = (
    unmatched
    .merge(
        spotify_exact_canonical[
            [
                "title_key",
                "artist_key",
                "name",
                "primary_artist",
            ]
        ],
        on="title_key",
        how="left",
        suffixes=(
            "_billboard",
            "_spotify",
        ),
    )
)

In [66]:
print(
    "Unmatched Billboard songs whose title exists somewhere in Spotify:",
    unmatched_title_check["name"].notna().sum()
)

print(
    "Unmatched Billboard songs whose title does NOT exist in Spotify:",
    unmatched_title_check["name"].isna().sum()
)

Unmatched Billboard songs whose title exists somewhere in Spotify: 122498
Unmatched Billboard songs whose title does NOT exist in Spotify: 10258


In [67]:
display(
    unmatched_title_check.loc[
        unmatched_title_check["name"].notna(),
        [
            "Song",
            "Artist",
            "primary_artist",
            "title_key",
        ]
    ].sample(
        30,
        random_state=42
    )
)

,Song,Artist,primary_artist,title_key
54593,Imagination,Xymox,Kate Smith,imagination
87086,Monster,"""Kanye West Featuring Jay-Z, Rick Ross, Bon Iv...",George Clanton,monster
41459,You And Me,Rockie Robbins,Tessanne Chin,you and me
87003,Memories,David Guetta Featuring Kid Cudi,Rella,memories
66132,Lullaby,Shawn Mullins,JJ Grey & Mofro,lullaby
43361,Memory,Barbra Streisand,Mortal Love,memory
108616,Nightmare,Halsey,DADDY LONG LEGS,nightmare
82997,One And The Same,Selena Gomez & Demi Lovato,Sam Lewis,one and the same
4829,Blue Christmas,The Browns Featuring Jim Edward Brown,Drew Holcomb & The Neighbors,blue christmas
38608,Under The Boardwalk,Billy Joe Royal,Rickie Lee Jones,under the boardwalk


In [68]:
spotify_title_keys = set(
    spotify_exact_canonical["title_key"]
)

unmatched["title_exists_in_spotify"] = (
    unmatched["title_key"]
    .isin(spotify_title_keys)
)

print(
    "Unmatched Billboard songs whose title exists somewhere in Spotify:",
    unmatched["title_exists_in_spotify"].sum()
)

print(
    "Unmatched Billboard songs whose title does NOT exist in Spotify:",
    (~unmatched["title_exists_in_spotify"]).sum()
)

print(
    "Total unmatched:",
    len(unmatched)
)

Unmatched Billboard songs whose title exists somewhere in Spotify: 11868
Unmatched Billboard songs whose title does NOT exist in Spotify: 10258
Total unmatched: 22126


In [69]:
spotify_title_counts = (
    spotify_exact_canonical
    .groupby("title_key")
    .size()
    .rename("spotify_title_candidates")
)

same_title_unmatched = (
    unmatched.loc[
        unmatched["title_exists_in_spotify"]
    ]
    .copy()
)

same_title_unmatched["spotify_title_candidates"] = (
    same_title_unmatched["title_key"]
    .map(spotify_title_counts)
)

In [70]:
display(
    same_title_unmatched[
        "spotify_title_candidates"
    ]
    .value_counts()
    .sort_index()
    .head(20)
)

spotify_title_candidates
1     3112
2     1603
3     1044
4      757
5      618
6      493
7      388
8      323
9      284
10     228
11     202
12     208
13     156
14     157
15     124
16     122
17     124
18     117
19      90
20      93
Name: count, dtype: int64

In [71]:
single_title_unmatched = (
    same_title_unmatched.loc[
        same_title_unmatched["spotify_title_candidates"] == 1
    ]
    .copy()
)

In [72]:
single_title_matches = (
    single_title_unmatched
    .merge(
        spotify_exact_canonical[
            [
                "title_key",
                "artist_key",
                "name",
                "primary_artist",
            ]
        ],
        on="title_key",
        how="left",
        suffixes=(
            "_billboard",
            "_spotify",
        ),
    )
)

In [73]:
display(
    single_title_matches[
        [
            "Song",
            "Artist",
            "primary_artist",
            "artist_key_billboard",
            "artist_key_spotify",
        ]
    ]
    .sample(
        30,
        random_state=42,
    )
)

,Song,Artist,primary_artist,artist_key_billboard,artist_key_spotify
557,(You Don't Have To) Paint Me A Picture,Gary Lewis And The Playboys,Alan Jackson,gary lewis and the playboys,alan jackson
416,Thou Shalt Not Steal,Dick and DeeDee,Dick & Dee Dee,dick and deedee,dick and dee dee
1690,Start The Car,Jude Cole,Travis Tritt,jude cole,travis tritt
2716,Lord Above,Fat Joe & Dre Featuring Eminem & Mary J. Blige,Fat Joe,fat joe and dre featuring eminem and mary j blige,fat joe
945,Ben,Michael Jackson,Zee Avi,michael jackson,zee avi
1494,Heaven In Your Eyes,Loverboy,Will Downing,loverboy,will downing
322,"""Something Old, Something New""",Paul and Paula,Lonnie Russ,paul and paula,lonnie russ
1809,Da' Dip,MC Luscious Featuring Kinsui,Freak Nasty,mc luscious featuring kinsui,freak nasty
2421,From Time,Drake Featuring Jhene Aiko,Drake,drake featuring jhene aiko,drake
1093,I Got Stoned And I Missed It,Jim Stafford,Dr. Hook,jim stafford,dr hook


In [74]:
def remove_featuring(artist_key):
    return re.sub(
        r"\s+featuring\s+.*$",
        "",
        artist_key
    ).strip()

In [75]:
same_title_unmatched["billboard_lead_artist_key"] = (
    same_title_unmatched["artist_key"]
    .apply(remove_featuring)
)

In [76]:
single_title_matches = (
    same_title_unmatched.loc[
        same_title_unmatched["spotify_title_candidates"] == 1
    ]
    .merge(
        spotify_exact_canonical[
            [
                "title_key",
                "artist_key",
                "name",
                "primary_artist",
            ]
        ],
        on="title_key",
        how="left",
        suffixes=(
            "_billboard",
            "_spotify",
        ),
    )
)

In [77]:
single_title_matches["lead_artist_match"] = (
    single_title_matches["billboard_lead_artist_key"]
    == single_title_matches["artist_key_spotify"]
)

print(
    single_title_matches["lead_artist_match"]
    .value_counts()
)

lead_artist_match
False    2784
True      328
Name: count, dtype: int64


In [78]:
display(
    single_title_matches.loc[
        single_title_matches["lead_artist_match"],
        [
            "Song",
            "Artist",
            "primary_artist",
            "billboard_lead_artist_key",
            "artist_key_spotify",
        ]
    ]
    .sample(
        30,
        random_state=42,
    )
)

,Song,Artist,primary_artist,billboard_lead_artist_key,artist_key_spotify
2415,Popular Song,Mika Featuring Ariana Grande,MIKA,mika,mika
2085,Enough Cryin,Mary J. Blige Featuring Brook-Lyn,Mary J. Blige,mary j blige,mary j blige
2479,One Man Can Change The World,Big Sean Featuring Kanye West & John Legend,Big Sean,big sean,big sean
1175,While I'm Alone,Maze Featuring Frankie Beverly,Maze,maze,maze
2047,What U Gon' Do,Lil Jon & The East Side Boyz Featuring Lil Scr...,Lil Jon & The East Side Boyz,lil jon and the east side boyz,lil jon and the east side boyz
2392,#thatPOWER,will.i.am Featuring Justin Bieber,will.i.am,will i am,will i am
2729,Be A Light,"""Thomas Rhett Featuring Reba McEntire, Hillary...",Thomas Rhett,thomas rhett,thomas rhett
2336,We'll Be Fine,Drake Featuring Birdman,Drake,drake,drake
2354,Right By My Side,Nicki Minaj Featuring Chris Brown,Nicki Minaj,nicki minaj,nicki minaj
2066,Badd,Ying Yang Twins Featuring Mike Jones & Mr. Col...,Ying Yang Twins,ying yang twins,ying yang twins


In [79]:
featured_matches = (
    single_title_matches.loc[
        single_title_matches["lead_artist_match"]
    ]
    .copy()
)

featured_matches["match_method"] = (
    "exact_title_lead_artist_after_featuring"
)

print(
    "Accepted featuring-format matches:",
    len(featured_matches)
)

Accepted featuring-format matches: 328


# Appendix — preserved thought process / retired exploration

These cells are intentionally commented out. They document how the current decisions were reached but are no longer required for the main pipeline.

In [52]:
# --- Historical exploration: previewing the huge raw CSV ---
#
# song_preview = pd.read_csv(
#     SONGS_PATH,
#     nrows=5,
# )
#
# print("Columns:", song_preview.columns.tolist())
# display(song_preview)

In [53]:
# --- Historical exploration: suspicious release years ---
#
# songs["year"].value_counts().sort_index()
#
# print("Earliest year:", songs["year"].min())
# print("Latest year:", songs["year"].max())
#
# # This revealed obvious errors such as modern songs assigned to 1900.
# # Decision: do not use the 550K `year` field as the forecasting timeline.
#
# display(
#     songs.loc[
#         songs["year"] < 1920,
#         ["name", "artists", "year", "genre", "niche_genres"],
#     ]
# )

In [54]:
# --- Historical exploration: auditing the source Folk category ---
#
# display(
#     songs.loc[
#         songs["genre"] == "Folk",
#         ["name", "artists", "genre", "niche_genres"],
#     ]
# )
#
# # This is where systematic reggae/dancehall misclassification became obvious.

In [55]:
# --- Historical exploration: official genre + full niche-combination counts ---
#
# genre_combo_counts = (
#     songs
#     .groupby(
#         ["genre", "niche_genres"],
#         dropna=False,
#     )
#     .size()
#     .reset_index(name="count")
#     .sort_values("count", ascending=False)
# )
#
# display(genre_combo_counts.head(50))
#
# genre_combo_counts.to_csv(
#     DATA_PROCESSED / "genre_combo_counts.csv",
#     index=False,
# )

In [56]:
# --- Historical exploration: initial "covered vs uncovered" override audit ---
#
# # Important lesson:
# # "Not explicitly covered by an override" does NOT mean "Other."
# # Unmapped niche tags fall back to the mapped official broad genre.
#
# override_tags = set().union(*GENRE_VOTE_TAGS.values())
#
# covered_tags = all_niche_tags & override_tags
# uncovered_tags = all_niche_tags - override_tags
#
# print("Total niche genres:", len(all_niche_tags))
# print("Covered by explicit votes:", len(covered_tags))
# print("Using fallback/inherit/ignore logic:", len(uncovered_tags))

In [57]:
# --- Historical exploration: dominant official genre by niche tag ---
#
# tag_genre_counts = (
#     songs[["genre_original", "niche_genre_list"]]
#     .explode("niche_genre_list")
#     .dropna(subset=["niche_genre_list"])
#     .assign(
#         niche_genre=lambda x:
#         x["niche_genre_list"]
#         .str.lower()
#         .str.strip()
#     )
#     .groupby(["niche_genre", "genre_original"])
#     .size()
#     .reset_index(name="count")
# )
#
# tag_totals = (
#     tag_genre_counts
#     .groupby("niche_genre")["count"]
#     .sum()
#     .rename("total")
# )
#
# dominant_genres = (
#     tag_genre_counts
#     .sort_values("count", ascending=False)
#     .drop_duplicates("niche_genre")
#     .merge(tag_totals, on="niche_genre")
# )
#
# dominant_genres["dominant_share"] = (
#     dominant_genres["count"]
#     / dominant_genres["total"]
# )
#
# # Practical review screen used during exploration:
# # high-volume tags + dominant share below 80%.
# # Low agreement alone does NOT mean the tag is wrong.

In [58]:
# --- Historical exploration: rebuilding genre_tag_audit.csv ---
#
# exploded_genres = (
#     songs[
#         [
#             "name",
#             "artists",
#             "genre_original",
#             "niche_genre_list",
#         ]
#     ]
#     .explode("niche_genre_list")
#     .dropna(subset=["niche_genre_list"])
#     .rename(columns={"niche_genre_list": "niche_genre"})
# )
#
# exploded_genres["niche_genre"] = (
#     exploded_genres["niche_genre"]
#     .str.lower()
#     .str.strip()
# )
#
# genre_distribution = (
#     exploded_genres
#     .groupby(["niche_genre", "genre_original"])
#     .size()
#     .unstack(fill_value=0)
# )
#
# genre_shares = genre_distribution.div(
#     genre_distribution.sum(axis=1),
#     axis=0,
# )
#
# genre_distribution.columns = [
#     f"{col}_count"
#     for col in genre_distribution.columns
# ]
#
# genre_shares.columns = [
#     f"{col}_share"
#     for col in genre_shares.columns
# ]
#
# genre_audit = pd.concat(
#     [genre_distribution, genre_shares],
#     axis=1,
# )
#
# genre_audit["total_count"] = (
#     genre_distribution.sum(axis=1)
# )
#
# official_counts = (
#     exploded_genres
#     .groupby(["niche_genre", "genre_original"])
#     .size()
#     .reset_index(name="count")
# )
#
# official_counts["rank"] = (
#     official_counts
#     .groupby("niche_genre")["count"]
#     .rank(method="first", ascending=False)
# )
#
# top1 = (
#     official_counts[official_counts["rank"] == 1]
#     .set_index("niche_genre")
# )
#
# top2 = (
#     official_counts[official_counts["rank"] == 2]
#     .set_index("niche_genre")
# )
#
# genre_audit["dominant_genre"] = top1["genre_original"]
# genre_audit["dominant_share"] = (
#     top1["count"] / genre_audit["total_count"]
# )
#
# genre_audit["second_genre"] = top2["genre_original"]
# genre_audit["second_share"] = (
#     top2["count"] / genre_audit["total_count"]
# )
#
#
# def get_examples(group, n=3):
#     examples = (
#         group[["name", "artists"]]
#         .drop_duplicates()
#         .head(n)
#     )
#
#     return " | ".join(
#         examples["name"].astype(str)
#         + " — "
#         + examples["artists"].astype(str)
#     )
#
#
# examples = (
#     exploded_genres
#     .groupby("niche_genre")[["name", "artists"]]
#     .apply(get_examples)
#     .rename("examples")
# )
#
# genre_audit = (
#     genre_audit
#     .join(examples)
#     .reset_index()
#     .sort_values("total_count", ascending=False)
# )
#
# genre_audit.to_csv(
#     DATA_PROCESSED / "genre_tag_audit.csv",
#     index=False,
# )

## Current stopping point

At this stage the notebook has:

- a clean Billboard popularity backbone,
- a memory-conscious 550K metadata load,
- documented rejection of the unreliable `year` field,
- a reproducible 15-category genre taxonomy,
- explicit high-confidence niche-tag overrides,
- safe fallback behavior for unmapped/ambiguous tags,
- audit columns preserving every classification decision,
- and commented matching code ready for the next stage.

**Recommended next action:** run the cleaned notebook through the genre audit, inspect a sample of changed rows, then freeze genre engineering and move to exact Billboard matching.